In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Saving the Infidels

In this notebook we want so solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missionaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.  If at any time the infidels outnumber the missionaries on one shore of the river, the missionaries are in danger.

We will encode this problem as a *constraint satisfaction problem*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossings of the river.  We use the following variables:
* $\texttt{M}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{B}i$ for $i\in\{0,\cdots,n\}$ is the number of boats on the western shore after the 
  $i^{\textrm{th}}$ crossing.

## Auxiliary Functions

The function `start` takes three integers as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns `true` in the initial state where everybody is on the western shore.

In [2]:
let start (M: int) (C: int) (B: int) : bool =
    M = 3 && C = 3 && B = 1

The function `goal` takes three integers as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns `true` in the state where everybody is on the eastern shore
and hence nobody is left on the western shore.

In [3]:
let goal (M: int) (C: int) (B: int) : bool =
    M = 0 && C = 0 && B = 0

The function `invariant` takes three integers as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns `true` if there is no problem on either shore of the river.  
There is no problem if any one of the following conditions is true:
* There are no missionaries on the western side of the shore, i.e. 
  $\texttt{M} = 0$.  
  Then all missionaries are on the eastern side of the shore.
* All missionaries are on the western side of the shore, i.e. $\texttt{M} = 3$.
  Then there are no missionaries on the eastern side of the shore.
* The number of missionaries on the western side is the same as the number of 
  infidels on that side, i.e. $\texttt{M} = \texttt{C}$.  Then the numbers of 
  missionaries and infidels have to match on the eastern shore as well.

Furthermore, as there is just one boat, the number $\texttt{B}$ of boats is at most one.

In [4]:
let invariant (M: int) (C: int) (B: int) : bool =
    (M = 0 || M = 3 || M = C) && B <= 1

The function `transition` takes 6 arguments:
* `Mα` is the number of missionaries on the western shore before the crossing.
* `Cα` is the number of infidels on the western shore before the crossing.
* `Bα` is the number of boats on the western shore before the crossing. 
* `Mβ` is the number of missionaries on the western shore after the crossing.
* `Cβ` is the number of infidels on the western shore after the crossing.
* `Bβ` is the number of boats on the western shore after the crossing.

It returns `true` if the crossing is admissible, i.e. if the persons starting on one shore arrive at the opposite shore after the crossing.  The condition 
```
   1 <= Mα - Mβ + Cα - Cβ <= 2
```
ensures that the boat has at least one and at most two passengers when travelling from west to east.  The conditions
```
Mβ <= Mα and Cβ <= Cα
```
ensure that the number of missionaries and infidels on the western shore does not increase when the boat travels from the western shore to the eastern shore.

In [5]:
let transition (Mα: int) (Cα: int) (Bα: int) (Mβ: int) (Cβ: int) (Bβ: int) : bool =
    if Bβ <> 1 - Bα then        // boat has to change sides during a crossing
        false
    elif Bα = 1 then            // boat travels from west to east
        let passengers = Mα - Mβ + Cα - Cβ
        1 <= passengers && passengers <= 2 && Mβ <= Mα && Cβ <= Cα
    else                        // boat travels from east to west
        let passengers = Mβ - Mα + Cβ - Cα
        1 <= passengers && passengers <= 2 && Mβ >= Mα && Cβ >= Cα

We import the backtracking solver.  The notebook `02-Backtracking-Constraint-Solver.ipynb` defines the types `Constraint<'V>` and `CSP<'V>`, the function `constr` that creates constraints, and the function `solve`.

In [6]:
importNotebook "02-Backtracking-Constraint-Solver.ipynb"

imported 

02-Backtracking-Constraint-Solver.ipynb

The function `missionariesCSP` takes one argument:
- `n` is the number of crossings.

It creates a CSP that tries to solve the problem with `n` crossings.  We use the following auxiliary functions:
- `vars i` returns the list of the variables `Mi`, `Ci`, and `Bi` that describe the state after the $i^{\textrm{th}}$ crossing.
- `apply3 f A i` applies the function `f` to the values of the variables `Mi`, `Ci`, and `Bi` in the assignment `A`.  As F# functions are *curried*, `apply3 (apply3 transition A i) A (i+1)` applies `transition` to the six values describing the states before and after the crossing.
- `text name xs` returns a string of the form `name(x1, ..., xn)` that is used as the text of the constraint.

In [7]:
let missionariesCSP (n: int) : CSP<int> =
    let vars i = [ sprintf "M%d" i; sprintf "C%d" i; sprintf "B%d" i ]
    let apply3 (f: int -> int -> int -> 'R) (A: Map<string, int>) (i: int) : 'R =
        f A[sprintf "M%d" i] A[sprintf "C%d" i] A[sprintf "B%d" i]
    let text name xs = sprintf "%s(%s)" name (String.concat ", " xs)
    let Start = constr (text "start" (vars 0)) (vars 0) (fun A -> apply3 start A 0)
    let Goal  = constr (text "goal"  (vars n)) (vars n) (fun A -> apply3 goal  A n)
    let Rest  = [ for i in 0 .. n - 1 do
                    constr (text "invariant" (vars i)) (vars i) (fun A -> apply3 invariant A i)
                    constr (text "transition" (vars i @ vars (i+1))) (vars i @ vars (i+1))
                           (fun A -> apply3 (apply3 transition A i) A (i+1)) ]
    { Variables   = List.concat [ for i in 0 .. n -> vars i ]
      Values      = [ 0; 1; 2; 3 ]
      Constraints = Start :: Goal :: Rest }

In [8]:
missionariesCSP 3

{ Variables = ["M0"; "C0"; "B0"; "M1"; "C1"; "B1"; "M2"; "C2"; "B2"; "M3"; "C3"; "B3"]; Values = [0; 1; 2; 3]; Constraints = [start(M0, C0, B0); goal(M3, C3, B3); invariant(M0, C0, B0); transition(M0, C0, B0, M1, C1, B1); invariant(M1, C1, B1); transition(M1, C1, B1, M2, C2, B2); invariant(M2, C2, B2); transition(M2, C2, B2, M3, C3, B3)] }

The function `findSolution` takes no arguments.  It computes a solution to the problem of saving the infidels by trying $n = 1, 3, 5, \cdots$ crossings until a solution is found.  It returns the number of crossings `n` together with the solution.

In [9]:
let findSolution () : int * Map<string, int> =
    let rec search (n: int) =
        printfn "%d" n
        match solve (missionariesCSP n) with
        | Some solution -> (n, solution)
        | None          -> search (n + 2)
    search 1

In [10]:
#!time
let n, Solution = findSolution ()

1

3

5

7

9

11

Wall time: 874.4331ms

In [11]:
n, Solution

(11, map [("B0", 1); ("B1", 0); ("B10", 1); ("B11", 0); ("B2", 1); ("B3", 0); ("B4", 1); ("B5", 0); ("B6", 1); ("B7", 0); ("B8", 1); ("B9", 0); ("C0", 3); ("C1", 2); ("C10", 2); ("C11", 0); ("C2", 2); ("C3", 0); ("C4", 1); ("C5", 1); ("C6", 2); ("C7", 2); ("C8", 3); ("C9", 1); ("M0", 3); ("M1", 2); ("M10", 0); ("M11", 0); ("M2", 3); ("M3", 3); ("M4", 3); ("M5", 1); ("M6", 2); ("M7", 0); ("M8", 0); ("M9", 0)])

The function `showSolution` takes two arguments:
- `Solution` is a solution of the CSP,
- `n` is the number of crossings.

It prints the solution.  For every $i \in \{0, \cdots, n\}$, it prints the situation after the $i^{\textrm{th}}$ crossing.  The western shore is printed on the left, the eastern shore on the right.  Between two situations, the passengers of the boat are shown together with the direction of the crossing.

In [12]:
let showSolution (Solution: Map<string, int>) (n: int) : unit =
    let rep (k: int) (s: string) = String.replicate k s
    for i in 0 .. n do
        let M = Solution[sprintf "M%d" i]
        let C = Solution[sprintf "C%d" i]
        let B = Solution[sprintf "B%d" i]
        printfn "%s%s%s%s%s" (rep M "😇") (rep C "🥷") (String.replicate 28 " ") (rep (3 - M) "😇") (rep (3 - C) "🥷")
        if i < n then
            if B = 1 then
                let MB = M - Solution[sprintf "M%d" (i+1)]
                let CB = C - Solution[sprintf "C%d" (i+1)]
                printfn "%s>>> %s%s >>>" (String.replicate 12 " ") (rep MB "😇") (rep CB "🥷")
            else
                let MB = Solution[sprintf "M%d" (i+1)] - M
                let CB = Solution[sprintf "C%d" (i+1)] - C
                printfn "%s<<< %s%s <<<" (String.replicate 12 " ") (rep MB "😇") (rep CB "🥷")

In [13]:
showSolution Solution n

😇😇😇

🥷🥷🥷

>>> 

😇

🥷

...

>>> 

🥷🥷

 >>>

😇😇😇

🥷🥷🥷